# Learning Objectives
In this notebook, you will learn Spark Dataframe APIs.

# Question List

Solve the following questions using Spark Dataframe APIs

### Join

1. easy - https://pgexercises.com/questions/joins/simplejoin.html
2. easy - https://pgexercises.com/questions/joins/simplejoin2.html
3. easy - https://pgexercises.com/questions/joins/self2.html 
4. medium - https://pgexercises.com/questions/joins/threejoin.html (three join)
5. medium - https://pgexercises.com/questions/joins/sub.html (subquery and join)

### Aggregation

1. easy - https://pgexercises.com/questions/aggregates/count3.html Group by order by
2. easy - https://pgexercises.com/questions/aggregates/fachours.html group by order by
3. easy - https://pgexercises.com/questions/aggregates/fachoursbymonth.html group by with condition 
4. easy - https://pgexercises.com/questions/aggregates/fachoursbymonth2.html group by multi col
5. easy - https://pgexercises.com/questions/aggregates/members1.html count distinct
6. med - https://pgexercises.com/questions/aggregates/nbooking.html group by multiple cols, join

### String & Date

1. easy - https://pgexercises.com/questions/string/concat.html format string
2. easy - https://pgexercises.com/questions/string/case.html WHERE + string function
3. easy - https://pgexercises.com/questions/string/reg.html WHERE + string function
4. easy - https://pgexercises.com/questions/string/substr.html group by, substr
5. easy - https://pgexercises.com/questions/date/series.html generate ts
6. easy - https://pgexercises.com/questions/date/bookingspermonth.html extract month from ts

### Question

How can you produce a list of the start times for bookings by members named 'David Farrell'?

https://pgexercises.com/questions/joins/simplejoin.html

In [0]:
# Write you solution here
# hint: you might need to re-run `0 - ETL pgexercieses CSV files` notebook to init tables

df = spark.sql("""
    SELECT b.starttime
    FROM bookings b
    JOIN members m ON b.memid = m.memid
    WHERE m.firstname = 'David' AND m.surname = 'Farrell'
""")
display(df)

starttime
2012-09-18T09:00:00.000Z
2012-09-18T17:30:00.000Z
2012-09-18T13:30:00.000Z
2012-09-18T20:00:00.000Z
2012-09-19T09:30:00.000Z
2012-09-19T15:00:00.000Z
2012-09-19T12:00:00.000Z
2012-09-20T15:30:00.000Z
2012-09-20T11:30:00.000Z
2012-09-20T14:00:00.000Z


### Question

How can you produce a list of the start times for bookings for tennis courts, for the date '2012-09-21'? Return a list of start time and facility name pairings, ordered by the time.

https://pgexercises.com/questions/joins/simplejoin2.html

In [0]:
df = spark.sql("""
    SELECT b.starttime, f.name
    FROM bookings b
    JOIN facilities f ON b.facid = f.facid
    WHERE f.name LIKE 'Tennis Court%'
      AND CAST(b.starttime AS DATE) = '2012-09-21'
    ORDER BY b.starttime
""")
display(df)

starttime,name
2012-09-21T08:00:00.000Z,Tennis Court 2
2012-09-21T08:00:00.000Z,Tennis Court 1
2012-09-21T09:30:00.000Z,Tennis Court 1
2012-09-21T10:00:00.000Z,Tennis Court 2
2012-09-21T11:30:00.000Z,Tennis Court 2
2012-09-21T12:00:00.000Z,Tennis Court 1
2012-09-21T13:30:00.000Z,Tennis Court 1
2012-09-21T14:00:00.000Z,Tennis Court 2
2012-09-21T15:30:00.000Z,Tennis Court 1
2012-09-21T16:00:00.000Z,Tennis Court 2


### Question

How can you output a list of all members, including the individual who recommended them (if any)? Ensure that results are ordered by (surname, firstname).

https://pgexercises.com/questions/joins/self2.html


In [0]:
df = spark.sql("""
    SELECT m.firstname, m.surname,
           r.firstname AS rec_firstname, r.surname AS rec_surname
    FROM members m
    LEFT JOIN members r ON m.recommendedby = r.memid
    ORDER BY m.surname, m.firstname
""")
display(df)

firstname,surname,rec_firstname,rec_surname
Florence,Bader,Ponder,Stibbons
Anne,Baker,Ponder,Stibbons
Timothy,Baker,Jemima,Farrell
Tim,Boothe,Tim,Rownam
Gerald,Butters,Darren,Smith
Joan,Coplin,Timothy,Baker
Erica,Crumpet,Tracy,Smith
Nancy,Dare,Janice,Joplette
David,Farrell,null,null
Jemima,Farrell,null,null


###Question
How can you produce a list of all members who have used a tennis court? Include in your output the name of the court, and the name of the member formatted as a single column. Ensure no duplicate data, and order by the member name followed by the facility name.

https://pgexercises.com/questions/joins/threejoin.html


In [0]:
df = spark.sql("""
    SELECT DISTINCT
        CONCAT(m.firstname, ' ', m.surname) AS member,
        f.name AS facility
    FROM bookings b
    JOIN members m ON b.memid = m.memid
    JOIN facilities f ON b.facid = f.facid
    WHERE f.name LIKE 'Tennis Court%'
      AND m.memid != 0
    ORDER BY member, facility
""")
display(df)

member,facility
Anne Baker,Tennis Court 1
Anne Baker,Tennis Court 2
Burton Tracy,Tennis Court 1
Burton Tracy,Tennis Court 2
Charles Owen,Tennis Court 1
Charles Owen,Tennis Court 2
Darren Smith,Tennis Court 2
David Farrell,Tennis Court 1
David Farrell,Tennis Court 2
David Jones,Tennis Court 1


###Question
How can you output a list of all members, including the individual who recommended them (if any), without using any joins? Ensure that there are no duplicates in the list, and that each firstname + surname pairing is formatted as a column and ordered.

https://pgexercises.com/questions/joins/sub.html


In [0]:
df = spark.sql("""
    SELECT DISTINCT
        CONCAT(m.firstname, ' ', m.surname) AS member,
        (SELECT MAX(CONCAT(r.firstname, ' ', r.surname))
         FROM members r
         WHERE r.memid = m.recommendedby) AS recommender
    FROM members m
    ORDER BY member, recommender
""")
display(df)

member,recommender
Anna Mackenzie,Darren Smith
Anne Baker,Ponder Stibbons
Burton Tracy,null
Charles Owen,Darren Smith
Darren Smith,null
David Farrell,null
David Jones,Janice Joplette
David Pinker,Jemima Farrell
Douglas Jones,David Jones
Erica Crumpet,Tracy Smith


###Question
Produce a count of the number of recommendations each member has made. Order by member ID.

https://pgexercises.com/questions/aggregates/count3.html


In [0]:
df = spark.sql("""
    SELECT recommendedby, COUNT(*) AS count
    FROM members
    WHERE recommendedby IS NOT NULL
    GROUP BY recommendedby
    ORDER BY recommendedby
""")
display(df)

recommendedby,count
1,5
2,3
3,1
4,2
5,1
6,1
9,2
11,1
13,2
15,1


###Question
Produce a list of the total number of slots booked per facility. For now, just produce an output table consisting of facility id and slots, sorted by facility id.

https://pgexercises.com/questions/aggregates/fachours.html



In [0]:
df = spark.sql("""
    SELECT facid, SUM(slots) AS total_slots
    FROM bookings
    GROUP BY facid
    ORDER BY facid
""")
display(df)

facid,total_slots
0,1320
1,1278
2,1209
3,830
4,1404
5,228
6,1104
7,908
8,911


###Question
Produce a list of the total number of slots booked per facility in the month of September 2012. Produce an output table consisting of facility id and slots, sorted by the number of slots.

https://pgexercises.com/questions/aggregates/fachoursbymonth.html


In [0]:
df = spark.sql("""
    SELECT facid, SUM(slots) AS total_slots
    FROM bookings
    WHERE starttime >= '2012-09-01' AND starttime < '2012-10-01'
    GROUP BY facid
    ORDER BY total_slots
""")
display(df)

facid,total_slots
5,122
3,422
7,426
8,471
6,540
2,570
1,588
0,591
4,648


###Question
Produce a list of the total number of slots booked per facility per month in the year of 2012. Produce an output table consisting of facility id and slots, sorted by the id and month.

https://pgexercises.com/questions/aggregates/fachoursbymonth2.html



In [0]:
df = spark.sql("""
    SELECT facid, MONTH(starttime) AS month, SUM(slots) AS total_slots
    FROM bookings
    WHERE YEAR(starttime) = 2012
    GROUP BY facid, MONTH(starttime)
    ORDER BY facid, month
""")
display(df)

facid,month,total_slots
0,7,270
0,8,459
0,9,591
1,7,207
1,8,483
1,9,588
2,7,180
2,8,459
2,9,570
3,7,104


###Question
Find the total number of members (including guests) who have made at least one booking.

https://pgexercises.com/questions/aggregates/members1.html


In [0]:
df = spark.sql("""
    SELECT COUNT(DISTINCT memid) AS member_count
    FROM bookings
""")
display(df)

member_count
30


###Question
Produce a list of each member name, id, and their first booking after September 1st 2012. Order by member ID.

https://pgexercises.com/questions/aggregates/nbooking.html


In [0]:
df = spark.sql("""
    SELECT m.memid, m.firstname, m.surname, MIN(b.starttime) AS first_booking
    FROM members m
    JOIN bookings b ON m.memid = b.memid
    WHERE b.starttime >= '2012-09-01'
    GROUP BY m.memid, m.firstname, m.surname
    ORDER BY m.memid
""")
display(df)

memid,firstname,surname,first_booking
0,GUEST,GUEST,2012-09-01T08:00:00.000Z
1,Darren,Smith,2012-09-01T09:00:00.000Z
2,Tracy,Smith,2012-09-01T11:30:00.000Z
3,Tim,Rownam,2012-09-01T16:00:00.000Z
4,Janice,Joplette,2012-09-01T15:00:00.000Z
5,Gerald,Butters,2012-09-02T12:30:00.000Z
6,Burton,Tracy,2012-09-01T15:00:00.000Z
7,Nancy,Dare,2012-09-01T12:30:00.000Z
8,Tim,Boothe,2012-09-01T08:30:00.000Z
9,Ponder,Stibbons,2012-09-01T11:00:00.000Z


###Question
Output the names of all members, formatted as 'Surname, Firstname'

https://pgexercises.com/questions/string/concat.html


In [0]:
df = spark.sql("""
    SELECT CONCAT(surname, ', ', firstname) AS name
    FROM members
""")
display(df)

name
"GUEST, GUEST"
"Smith, Darren"
"Smith, Tracy"
"Rownam, Tim"
"Joplette, Janice"
"Butters, Gerald"
"Tracy, Burton"
"Dare, Nancy"
"Boothe, Tim"
"Stibbons, Ponder"


###Question
Perform a case-insensitive search to find all facilities whose name begins with 'tennis'. Retrieve all columns.

https://pgexercises.com/questions/string/case.html

In [0]:
df = spark.sql("""
    SELECT *
    FROM facilities
    WHERE LOWER(name) LIKE 'tennis%'
""")
display(df)

facid,name,membercost,guestcost,initialoutlay,monthlymaintenance
0,Tennis Court 1,5.0,25.0,10000.0,200.0
1,Tennis Court 2,5.0,25.0,8000.0,200.0


###Question
You've noticed that the club's member table has telephone numbers with very inconsistent formatting. You'd like to find all the telephone numbers that contain parentheses, returning the member ID and telephone number sorted by member ID.

https://pgexercises.com/questions/string/reg.html

In [0]:
df = spark.sql("""
    SELECT memid, telephone
    FROM members
    WHERE telephone LIKE '%(%' OR telephone LIKE '%)%'
    ORDER BY memid
""")
display(df)

memid,telephone
0,(000) 000-0000
3,(844) 693-0723
4,(833) 942-4710
5,(844) 078-4130
6,(822) 354-9973
7,(833) 776-4001
8,(811) 433-2547
9,(833) 160-3900
10,(855) 542-5251
11,(844) 536-8036


###Question
You'd like to produce a count of how many members you have whose surname starts with each letter of the alphabet. Sort by the letter, and don't worry about printing out a letter if the count is 0.

https://pgexercises.com/questions/string/substr.html


In [0]:
df = spark.sql("""
    SELECT UPPER(SUBSTRING(surname, 1, 1)) AS letter, COUNT(*) AS count
    FROM members
    GROUP BY UPPER(SUBSTRING(surname, 1, 1))
    ORDER BY letter
""")
display(df)

letter,count
B,5
C,2
D,1
F,2
G,2
H,1
J,3
M,1
O,1
P,2


###Question
Produce a list of all the dates in October 2012. They can be output as a timestamp (with time set to midnight) or a date.

https://pgexercises.com/questions/date/series.html


In [0]:
df = spark.sql("""
    SELECT explode(sequence(to_date('2012-10-01'), to_date('2012-10-31'), interval 1 day)) AS date
""")
display(df)

date
2012-10-01
2012-10-02
2012-10-03
2012-10-04
2012-10-05
2012-10-06
2012-10-07
2012-10-08
2012-10-09
2012-10-10


###Question
Return a count of bookings for each month, sorted by month

https://pgexercises.com/questions/date/bookingspermonth.html


In [0]:
df = spark.sql("""
    SELECT MONTH(starttime) AS month, COUNT(*) AS bookings_count
    FROM bookings
    WHERE YEAR(starttime) = 2012
    GROUP BY MONTH(starttime)
    ORDER BY month
""")
display(df)

month,bookings_count
7,658
8,1472
9,1913
